[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nursnaaz/zero-to-genai-engineer/blob/main/11_LangGraph/notebooks/02_human_in_the_loop_and_multi_agent.ipynb)

# Human-in-the-Loop & Multi-Agent Systems

Notebook 11 in Module 10 already built a **working** `interrupt()`-based approval flow —
`HumanInTheLoopMiddleware` on a `create_agent`, pausing before a risky tool call and resuming
with `Command(resume=...)`. This notebook is not "here's a feature S10 skipped" — it's "here's
the primitive that middleware is made of," because the capstone in Notebook 03 needs
human-in-the-loop for a decision no prebuilt middleware could package (escalating after a
custom multi-step retry loop, not gating a single named tool). Once you've built the raw
mechanism by hand once, both uses — the packaged middleware and the bespoke escalation node —
read as the same four moving parts: `interrupt()`, a payload, `Command(resume=...)`, and a
checkpointer to hold the pause.

## What you'll build today

1. **`interrupt()` + `Command(resume=...)`** — pause a tool call for human approval, built
   from scratch on a hand-written `StateGraph` (S10f used the same mechanism packaged as
   `HumanInTheLoopMiddleware`)
2. **Long-term memory (`Store`)** — facts that persist *across* different conversations, not
   just within one (S10f used the same `Store` with semantic search via `@dynamic_prompt`;
   here it's a plain key lookup, the minimum needed to see the mechanism clearly)
3. **Multi-agent supervisor** — one router agent handing work to specialists via
   `Command(goto=...)` — genuinely new relative to S10f, which didn't cover multi-agent
   coordination

## 0. Setup

In [ ]:
%pip install -q langgraph>=0.6 langchain>=1.0 langchain-openai python-dotenv

import warnings, os
warnings.filterwarnings("ignore")
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent.parent / ".env")
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1. `interrupt()` — pausing for human approval

The scenario: an agent can check the weather freely, but sending an email needs a human to
approve it first. We route tool calls through a `human_approval` node *only* when the
requested tool is the risky one.

`interrupt(payload)` does three things in one call: it (1) freezes graph execution at that
exact point, (2) persists the current state via the checkpointer, and (3) returns `payload`
to whoever called `.invoke()`/`.stream()`, so they can show it to a human.

In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver


@tool
def get_weather(city: str) -> str:
    """Return the current weather for a city (simulated)."""
    return f"{city}: 29C, sunny"


@tool
def send_email(to: str, body: str) -> str:
    """Send an email (simulated)."""
    return f"Email sent to {to}: {body[:50]}..."


risky_tools = {"send_email"}
tools = [get_weather, send_email]
llm_with_tools = llm.bind_tools(tools)


def call_model(state: MessagesState) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


def human_approval(state: MessagesState) -> Command:
    last = state["messages"][-1]
    tool_call = last.tool_calls[0]
    decision = interrupt({
        "reason": "This tool call needs human approval before it runs.",
        "tool": tool_call["name"],
        "args": tool_call["args"],
    })
    if decision.get("approved"):
        return Command(goto="tools")
    rejection = ToolMessage(content="Rejected by human reviewer.", tool_call_id=tool_call["id"])
    return Command(goto="agent", update={"messages": [rejection]})


def route_after_model(state: MessagesState) -> str:
    last = state["messages"][-1]
    if not getattr(last, "tool_calls", None):
        return END
    return "human_approval" if last.tool_calls[0]["name"] in risky_tools else "tools"


hitl_builder = StateGraph(MessagesState)
hitl_builder.add_node("agent", call_model)
hitl_builder.add_node("human_approval", human_approval)
hitl_builder.add_node("tools", ToolNode(tools))
hitl_builder.add_edge(START, "agent")
hitl_builder.add_conditional_edges(
    "agent", route_after_model, {"human_approval": "human_approval", "tools": "tools", END: END}
)
hitl_builder.add_edge("tools", "agent")

hitl_app = hitl_builder.compile(checkpointer=InMemorySaver())

Run it. The graph will stop *inside* `human_approval` and hand back an `__interrupt__` payload
instead of a final answer — that's the pause.

In [ ]:
config = {"configurable": {"thread_id": "approval-demo"}}
result = hitl_app.invoke(
    {"messages": [("user", "Email jane@example.com to confirm tomorrow's 10am meeting.")]},
    config,
)

if "__interrupt__" in result:
    payload = result["__interrupt__"][0].value
    print("PAUSED -- waiting for a human:")
    print(" ", payload)
else:
    result["messages"][-1].pretty_print()

Resume it with `Command(resume=...)`, on the **same `thread_id`** — the graph continues from
inside `human_approval`, not from the top.

In [ ]:
approved = hitl_app.invoke(Command(resume={"approved": True}), config)
approved["messages"][-1].pretty_print()

# Try it again on a fresh thread, but reject this time:
config2 = {"configurable": {"thread_id": "approval-demo-2"}}
hitl_app.invoke({"messages": [("user", "Email bob@example.com telling him he's fired.")]}, config2)
rejected = hitl_app.invoke(Command(resume={"approved": False}), config2)
rejected["messages"][-1].pretty_print()

## 2. Long-term memory — facts that outlive one conversation

`checkpointer` (Notebook 01) gives you memory *within* one `thread_id`. A `Store` gives you
memory that any thread can read or write — a real "remember this about the user, forever"
mechanism. Nodes get the store injected automatically when you declare a `store=` keyword-only
parameter.

**A subtle bug worth seeing on purpose:** if the *same* node both writes a preference AND
runs on every turn, a later thread that just says "Hello" will silently **overwrite** the
earlier preference with a default before it ever gets a chance to read it back — the demo
would then "prove" recall by showing you the value it just wrote a moment earlier, not
anything actually remembered. The fix is the one real rule of long-term memory: **separate
the write path from the read path.** Below, `set_preference` only runs when a preference is
being explicitly stated; `greet_with_tone` only ever reads.

In [ ]:
from langgraph.store.memory import InMemoryStore
from langchain_core.messages import AIMessage

store = InMemoryStore()


def set_preference(state: MessagesState, *, store) -> dict:
    """Write path -- runs only when the user is explicitly stating a preference."""
    store.put(("preferences", "user-42"), "tone", {"value": "casual"})
    return {"messages": [AIMessage(content="Got it, I'll keep things casual from now on.")]}


def greet_with_tone(state: MessagesState, *, store) -> dict:
    """Read path -- never writes, so it can't clobber what a different thread stored."""
    item = store.get(("preferences", "user-42"), "tone")
    tone = item.value["value"] if item else "neutral"
    greeting = ("Hey! What's up?" if tone == "casual"
                else "Good day. How may I assist you?")
    return {"messages": [AIMessage(content=f"[{tone} tone recalled] {greeting}")]}


pref_builder = StateGraph(MessagesState)
pref_builder.add_node("set_preference", set_preference)
pref_builder.add_edge(START, "set_preference")
pref_builder.add_edge("set_preference", END)
pref_app = pref_builder.compile(store=store)

greet_builder = StateGraph(MessagesState)
greet_builder.add_node("greet_with_tone", greet_with_tone)
greet_builder.add_edge(START, "greet_with_tone")
greet_builder.add_edge("greet_with_tone", END)
greet_app = greet_builder.compile(store=store)   # <-- same `store` instance, different graph

# Thread 1, today: the user explicitly states a preference once.
pref_app.invoke(
    {"messages": [("user", "Please keep things casual with me.")]}, {"configurable": {"thread_id": "t1"}}
)
# A completely DIFFERENT thread, days later, that never mentioned any preference --
# still recalls it, because Store memory isn't scoped to a thread_id:
out = greet_app.invoke(
    {"messages": [("user", "Hello.")]}, {"configurable": {"thread_id": "t2-days-later"}}
)
out["messages"][-1].pretty_print()

## 3. Multi-agent supervisor

A supervisor node classifies the request and hands off to a specialist via
`Command(goto=...)`. Each specialist hands control back to the supervisor the same way. This
is the pattern behind the M10 curriculum line "supervisor pattern" — and it's built from
nothing more exotic than the `Command` you already used for the approval handoff above.

**A routing bug worth seeing on purpose:** it's tempting to have the supervisor re-classify
`state["messages"][-1]` on every visit. That breaks the moment a specialist hands control
back — `messages[-1]` is now *the specialist's own answer*, not a fresh request, so you're
asking the router to "route" a sentence like `"[math_agent] 611"`. It usually still limps to
a stop, but only because of a message-count safety valve doing the real work, not because the
routing logic is actually correct. The fix: **only re-classify when the last message is a
fresh `HumanMessage`.** The moment the last message is an `AIMessage` (a specialist just
answered), the supervisor's job for this turn is done.

In [ ]:
from langchain_core.messages import AIMessage


def supervisor(state: MessagesState) -> Command:
    last = state["messages"][-1]
    if isinstance(last, AIMessage):
        # A specialist already answered this request -- nothing left to route.
        return Command(goto=END)
    verdict = llm.invoke(
        "Route this request to exactly one worker: research_agent or math_agent.\n"
        f"Request: {last.content}\nAnswer with one word only."
    ).content.strip().upper()
    goto = "math_agent" if "MATH" in verdict else "research_agent"
    return Command(goto=goto)


def research_agent(state: MessagesState) -> Command:
    answer = llm.invoke(
        f"You are a research specialist. Briefly answer: {state['messages'][-1].content}"
    ).content
    return Command(goto="supervisor", update={"messages": [AIMessage(content=f"[research_agent] {answer}")]})


def math_agent(state: MessagesState) -> Command:
    answer = llm.invoke(
        f"You are a math specialist. Solve, showing the final number clearly: {state['messages'][-1].content}"
    ).content
    return Command(goto="supervisor", update={"messages": [AIMessage(content=f"[math_agent] {answer}")]})


supervisor_builder = StateGraph(MessagesState)
supervisor_builder.add_node("supervisor", supervisor)
supervisor_builder.add_node("research_agent", research_agent)
supervisor_builder.add_node("math_agent", math_agent)
supervisor_builder.add_edge(START, "supervisor")
supervisor_app = supervisor_builder.compile()

print(supervisor_app.get_graph().draw_mermaid())

for q in ["What is 47 * 13?", "What year was the Transformer paper published?"]:
    out = supervisor_app.invoke({"messages": [("user", q)]})
    print(q, "->", out["messages"][-1].content)

**Note on nodes that return `Command(goto=...)` directly:** you don't need
`add_conditional_edges` for these transitions — the routing decision travels *with* the
return value (verified above: `supervisor_builder` never declares an edge from `supervisor`
to either specialist, and the graph still routes correctly). This is exactly how the
batteries-included [`langgraph-supervisor`](https://github.com/langchain-ai/langgraph-supervisor-py)
package (`create_supervisor(...)`) builds the same pattern as a one-liner once you understand
what it compiles to.

## Summary

| Primitive | Gives you |
|---|---|
| `interrupt()` + `Command(resume=...)` | A graph that pauses, waits for a human, and resumes from the *exact* node that paused — not from the top |
| `Store` | Memory that survives across different `thread_id`s, not just within one |
| `Command(goto=...)` | A node that decides, at runtime, which node runs next — the basis of the supervisor pattern |

**Next: `03_agentic_rag_capstone.ipynb`** — all three of these, plus everything Module 10
built (hybrid retrieval, reranking, groundedness), combined into one self-correcting Agentic
RAG system.